# Customer Behavior Analysis

This notebook reproduces the analysis step by step, creates the figures, checks data quality, and explores additional hidden patterns. Place `CustomerBehavior.csv` in the same folder as this notebook before running it.

## 1. Imports and setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
from IPython.display import display

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


## 2. Load the data

In [ ]:
DATA_PATH = Path('CustomerBehavior.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError('Put CustomerBehavior.csv in the same folder as this notebook.')

df = pd.read_csv(DATA_PATH)
display(df.head())
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')


## 3. Data dictionary and quality checks

In [ ]:
df.info()
print('\nMissing values:')
display(df.isna().sum().to_frame('missing'))
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate customer IDs:', df['Customer ID'].duplicated().sum())
print('Unique values in categorical columns:')
for c in ['Gender','City','Membership Type','Discount Applied','Satisfaction Level']:
    print(f'\n{c}:', df[c].dropna().unique())


## 4. Cleaning and derived variables

In [ ]:
numeric_cols = ['Age','Total Spend','Items Purchased','Average Rating','Days Since Last Purchase']
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['Discount Applied'] = (df['Discount Applied'].astype(str).str.upper() == 'TRUE')
df['Spend per Item'] = df['Total Spend'] / df['Items Purchased']
df['Age Group'] = pd.cut(df['Age'], bins=[0,29,39,49,100], labels=['18–29','30–39','40–49','50+'])
df['Recency Group'] = pd.cut(df['Days Since Last Purchase'], bins=[-1,14,30,45,np.inf], labels=['0–14 days','15–30 days','31–45 days','46+ days'])
df['High Value'] = df['Total Spend'] >= df['Total Spend'].quantile(.75)
display(df.head())


## 5. Overall descriptive statistics

In [ ]:
display(df[numeric_cols + ['Spend per Item']].describe().T)
print('Total spend:', f'${df["Total Spend"].sum():,.2f}')
print('Satisfaction distribution:')
display(df['Satisfaction Level'].value_counts(dropna=False).rename('customers').to_frame())


## 6. Membership analysis

In [ ]:
membership = df.groupby('Membership Type', observed=True).agg(
    Customers=('Customer ID','count'),
    Total_Revenue=('Total Spend','sum'),
    Avg_Spend=('Total Spend','mean'),
    Median_Spend=('Total Spend','median'),
    Avg_Items=('Items Purchased','mean'),
    Avg_Rating=('Average Rating','mean'),
    Avg_Recency=('Days Since Last Purchase','mean'),
    Satisfaction_Rate=('Satisfaction Level', lambda s: (s == 'Satisfied').mean())
).sort_values('Avg_Spend', ascending=False)
display(membership)


## 7. Core figures

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))

sns.barplot(data=membership.reset_index(), x='Membership Type', y='Avg_Spend', ax=axes[0,0], hue='Membership Type', legend=False, palette='viridis')
axes[0,0].set_title('Average Spend by Membership')
axes[0,0].set_ylabel('Average spend')

sns.boxplot(data=df, x='Membership Type', y='Total Spend', ax=axes[0,1], hue='Membership Type', legend=False, palette='Set2')
axes[0,1].set_title('Spend Distribution by Membership')

sat = df['Satisfaction Level'].value_counts(dropna=False)
sns.barplot(x=sat.index.astype(str), y=sat.values, ax=axes[0,2], hue=sat.index.astype(str), legend=False, palette='Set1')
axes[0,2].set_title('Satisfaction Counts')
axes[0,2].set_ylabel('Customers')

sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', center=0, ax=axes[1,0])
axes[1,0].set_title('Correlation Heatmap')

sns.scatterplot(data=df, x='Days Since Last Purchase', y='Total Spend', hue='Membership Type', alpha=.65, ax=axes[1,1])
axes[1,1].set_title('Recency vs Spend')

sns.histplot(data=df, x='Total Spend', hue='Membership Type', kde=True, element='step', ax=axes[1,2])
axes[1,2].set_title('Spend Distribution')

plt.tight_layout()
plt.savefig('core_customer_behavior_figures.png', dpi=200, bbox_inches='tight')
plt.show()


## 8. City and discount analysis

In [ ]:
city = df.groupby('City').agg(
    Customers=('Customer ID','count'),
    Avg_Spend=('Total Spend','mean'),
    Avg_Rating=('Average Rating','mean'),
    Avg_Recency=('Days Since Last Purchase','mean'),
    Unsatisfied_Rate=('Satisfaction Level', lambda s: (s == 'Unsatisfied').mean())
).sort_values('Avg_Spend', ascending=False)
display(city)

discount = df.groupby('Discount Applied', observed=True).agg(
    Customers=('Customer ID','count'),
    Avg_Spend=('Total Spend','mean'),
    Avg_Items=('Items Purchased','mean'),
    Avg_Rating=('Average Rating','mean'),
    Avg_Recency=('Days Since Last Purchase','mean'),
    Satisfaction_Rate=('Satisfaction Level', lambda s: (s == 'Satisfied').mean())
)
display(discount)


## 9. Additional hidden detail: engineered customer segments

In [ ]:
# Cross-tab satisfaction by membership
sat_by_membership = pd.crosstab(df['Membership Type'], df['Satisfaction Level'], normalize='index') * 100
display(sat_by_membership.round(1))

# Cross-tab recency groups by membership
recency_by_membership = pd.crosstab(df['Membership Type'], df['Recency Group'], normalize='index') * 100
display(recency_by_membership.round(1))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
sns.heatmap(sat_by_membership, annot=True, fmt='.1f', cmap='RdYlGn', ax=axes[0])
axes[0].set_title('Satisfaction Mix (%)')

sns.heatmap(recency_by_membership, annot=True, fmt='.1f', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Recency Mix (%)')

sns.barplot(data=df, x='Age Group', y='Total Spend', hue='Age Group', legend=False, palette='crest', ax=axes[2])
axes[2].set_title('Average Spend by Age Group')
axes[2].set_ylabel('Average spend')

plt.tight_layout()
plt.savefig('hidden_pattern_figures.png', dpi=200, bbox_inches='tight')
plt.show()


## 10. Statistical tests and effect sizes

In [ ]:
# Kruskal-Wallis test: spend differs across membership tiers
samples = [g['Total Spend'].dropna() for _, g in df.groupby('Membership Type', observed=True)]
kw = stats.kruskal(*samples)
print('Kruskal-Wallis membership vs spend:', kw)

# Spearman correlations are robust for monotonic relationships
for x in ['Items Purchased','Average Rating','Days Since Last Purchase','Age']:
    r, p = stats.spearmanr(df[x], df['Total Spend'], nan_policy='omit')
    print(f'Spearman {x} vs spend: rho={r:.3f}, p={p:.4g}')

# Discount difference in average spend; Mann-Whitney U
with_discount = df.loc[df['Discount Applied'], 'Total Spend'].dropna()
without_discount = df.loc[~df['Discount Applied'], 'Total Spend'].dropna()
u = stats.mannwhitneyu(with_discount, without_discount, alternative='two-sided')
print('Discount vs no-discount Mann-Whitney:', u)
print('Discount average spend difference:', with_discount.mean() - without_discount.mean())


## 11. Outliers and data integrity

In [ ]:
outlier_rows = []
for c in numeric_cols:
    q1, q3 = df[c].quantile([.25,.75])
    iqr = q3 - q1
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    count = ((df[c] < low) | (df[c] > high)).sum()
    outlier_rows.append([c, q1, q3, low, high, count])
outliers = pd.DataFrame(outlier_rows, columns=['Variable','Q1','Q3','Lower fence','Upper fence','Outlier count'])
display(outliers)

# Check for suspiciously repeated records, excluding customer ID
repeat_counts = df.drop(columns='Customer ID').value_counts()
print('Repeated behavioral profiles:', (repeat_counts > 1).sum())
display(repeat_counts.head(10).to_frame('count'))


## 12. Practical conclusions

In [ ]:
print('Use these outputs to write the final conclusions:')
print(f"1. Gold average spend is ${membership.loc['Gold','Avg_Spend']:,.0f}; Bronze is ${membership.loc['Bronze','Avg_Spend']:,.0f}.")
print(f"2. Gold average rating is {membership.loc['Gold','Avg_Rating']:.2f}; Bronze is {membership.loc['Bronze','Avg_Rating']:.2f}.")
print(f"3. Gold average recency is {membership.loc['Gold','Avg_Recency']:.1f} days; Bronze is {membership.loc['Bronze','Avg_Recency']:.1f} days.")
print(f"4. Spend-items correlation: {df[['Total Spend','Items Purchased']].corr().iloc[0,1]:.3f}.")
print(f"5. Spend-rating correlation: {df[['Total Spend','Average Rating']].corr().iloc[0,1]:.3f}.")
print('Do not claim that membership, discounts, or ratings cause spend; the data is observational.')


## Notes

The dataset has no transaction date, so a true time-series line plot or cohort-retention analysis cannot be justified. The repeated behavioral profiles and missing satisfaction labels should be investigated before production reporting. For causal conclusions, run a controlled promotion or membership experiment.